# Custom Dataset：Kana (usb39 / semilearn 0.3.2)

本 Notebook 用于：
- 在 **usb39 (Python 3.10)** 环境中运行
- 使用你本地 editable 安装的 semilearn（`/root/autodl-tmp/Semi-supervised-learning`）
- 训练 Freematch + ViT/Resnet50 on 自定义 Kana 数据集

⚠️ 如果你看到 semilearn 的路径指向 `/envs/usb/lib/python3.8/site-packages/...`，说明你选错 kernel（还是 usb/py38），会导致 `NotImplementedError`。

## 0) 环境与源码路径检查（必须通过）

In [1]:
import os, sys
import semilearn

print('Python:', sys.version)
print('semilearn file:', semilearn.__file__)

# 强制确保你在 usb39 + 本地 repo 版本
expected_repo = '/root/autodl-tmp/Semi-supervised-learning'
if expected_repo not in semilearn.__file__:
    raise RuntimeError(
        "你现在的 kernel/环境不对：semilearn 不是从本地 repo 导入的。\n"
        f"当前：{semilearn.__file__}\n\n"
        "请在 Jupyter 右上角切换 Kernel 到：Python (usb39)，\n"
        "并确认你已在 usb39 环境里 `pip install -e /root/autodl-tmp/Semi-supervised-learning`。"
    )

print('✅ 环境检查通过：usb39 + 本地 semilearn 源码')

Python: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
semilearn file: /root/autodl-tmp/Semi-supervised-learning/semilearn/__init__.py
✅ 环境检查通过：usb39 + 本地 semilearn 源码


## 1) 配置：Freematch + ViT + Kana

- 数据目录：`/root/autodl-tmp/kana_raw_data_with_concentration_preprocessed_images`
- 3 类分类
- 标注数：12（每类 4 张）
- 小数据集：建议先缩短迭代验证全流程，然后再加大训练

⚠️ 注意：如果你用 ViT 预训练权重，必须确保路径存在；否则把 `use_pretrain=False`。

In [2]:
from semilearn import get_config

config_dict = {
    # algorithm / model
    'algorithm': 'freematch',
    'net': 'resnet50',
    'use_pretrain': False,
    
    'pretrain_path': '/root/.cache/torch/hub/checkpoints/vit_tiny_patch2_32_mlp_im_1k_32.pth',

    # dataset
    'dataset': 'kana',
    'data_dir': '/root/autodl-tmp/kana_circle_patches',
    'num_classes': 3,
    'num_labels': 3,  # 根据你实际标注样本数量调整（选取3/30/60，对应到每类有标签数量为1/10/20）
    'val_ratio': 0.2,  # 适当调整验证集占比

    # training schedule
    'epoch': 256,
    'num_train_iter': 500,
    'num_eval_iter': 50,
    'num_log_iter': 10,
    
    # optimizer
    'optim': 'AdamW',
    'lr': 1e-4,
    'layer_decay': 0.5,

    # batch
    'batch_size': 8,
    'eval_batch_size': 16,

    # Freematch specific（Freematch算法相关的配置）
    'hard_label': True,
    # 设置为 True 时，Freematch 使用硬标签（hard labels）。在半监督学习中，硬标签意味着无标签数据的伪标签是直接从模型的预测中选出的，而不是经过软化（softening）。这是标准的 Freematch 方法。
    'uratio': 10,
    # 设置标签数据与无标签数据的比率。`uratio` 为 1 表示每个有标签样本对应 1 个无标签样本。调整这个值可以控制无标签样本的权重。增加 `uratio` 会使无标签样本的影响更大。
    'p_cutoff': 0.7,
    # `p_cutoff` 是一个阈值，用于伪标签生成。在 Freematch 中，模型会为无标签数据预测一个概率分布，然后根据这个分布生成伪标签。当模型的预测概率大于该阈值时，才将其伪标签化。此值控制了伪标签的质量（即伪标签的可靠性），通常设置为 0.7。
    'T': 0.5,
    # `T` 是温度（temperature）参数，用于 softmax 函数中的温度控制。该参数决定了模型对无标签样本的预测概率分布的"平滑程度"。较低的 `T` 会使预测分布更尖锐，而较高的 `T` 会使其更加平坦。通常在 semi-supervised learning 中，较低的 `T` 用于提高伪标签的确定性。
    'ulb_loss_ratio': 0.1,
    # `ulb_loss_ratio` 控制无标签样本在总损失中的权重。在 Freematch 中，训练时会同时计算有标签数据和无标签数据的损失，`ulb_loss_ratio` 用来平衡这两部分损失。通常设置为 0.5，表示有标签和无标签的损失权重相等。如果数据量较少，可以适当减小这个比率以避免无标签样本的影响过大。
    'ema_m': 0.999,
    # `ema_m` 是指数滑动平均（Exponential Moving Average, EMA）衰减因子。EMA 用于平滑模型权重，以提高无标签数据伪标签生成的稳定性。较大的 `ema_m` 会让模型慢慢更新，更稳定，但反应速度较慢。常见的 `ema_m` 值为 0.999。
    'include_lb_to_ulb': True,
    # 如果设置为 `True`，表示将已标注数据（labeled data）也包含到无标签数据的伪标签生成过程中。即使是标注数据也可以作为无标签数据的一部分参与训练，这有助于改进训练效果。在某些应用场景中，这种做法可以提升性能，但需要小心使用，以避免标签泄露。


    # system
    'gpu': 0,
    'world_size': 1,
    'distributed': False,
    'num_workers': 2,

    # IMPORTANT: semilearn 需要的字段
    'amp': False,
    'seed': 0,
    
    #保存路径
    'save_dir':'./saved_results/kana_experiment',
    
    #禁用保存模型参数
    'save_model': False
}

import itertools

# 如果你没有预训练权重文件，就自动关掉
if config_dict.get('use_pretrain', False) and (not os.path.exists(config_dict['pretrain_path'])):
    print('⚠️ pretrain 权重不存在，自动改为 use_pretrain=False')
    config_dict['use_pretrain'] = False
    config_dict.pop('pretrain_path', None)

def make_save_name(cfg):
    label_num_per_class = cfg["num_labels"] / cfg["num_classes"]
    return (
        f'{cfg["algorithm"]}_{cfg["dataset"]}_{cfg["net"]}_labels{label_num_per_class}'
        f'_train{cfg["num_train_iter"]}_val{cfg["num_eval_iter"]}_test{cfg["num_eval_iter"]}'
        f'_classes{cfg["num_classes"]}_epoch{cfg["epoch"]}_lr{cfg["lr"]}_seed{cfg["seed"]}'
    )

def save_path_exists(cfg):
    save_name = make_save_name(cfg)
    return os.path.exists(os.path.join(cfg["save_dir"], save_name))

# 定义不同的变量值
num_labels_list = [3, 15, 30, 45]
seed_list = [0, 126, 2001, 2026, 159357, 654369, 650108, 528057, 20410, 188997]
net_list = ['resnet50', 'vit_tiny_patch2_32']
# 生成所有待跑组合（跳过已经存在的保存路径）
run_configs = []
for num_labels, seed, net in itertools.product(num_labels_list, seed_list, net_list):
    candidate = dict(config_dict)
    candidate["num_labels"] = num_labels
    candidate["seed"] = seed
    candidate["net"] = net
    candidate["save_name"] = make_save_name(candidate)
    if not save_path_exists(candidate):
        run_configs.append(candidate)

print(f"✅ 待运行组合数: {len(run_configs)}")


✅ 待运行组合数: 0


## 2) 构建 Algorithm（会自动构建 dataset + dataloader + optimizer）

如果这里能成功，说明：
- Kana dataset 已被正确注册 / 实现
- get_dataset 能跑通
- 你的 ViT timm 兼容修改生效

我们会把构建出来的 loader 取出来用于 Trainer。

In [3]:
from semilearn import get_algorithm, get_net_builder, Trainer

for i, cfg_dict in enumerate(run_configs, 1):
    cfg = get_config(cfg_dict)
    print(f"\n===== Run {i}/{len(run_configs)} =====")
    print(cfg.save_name)

    alg = get_algorithm(cfg, get_net_builder(cfg.net, from_name=False), tb_log=None, logger=None)
    print('✅ algorithm ok:', type(alg).__name__)

    loader_dict = getattr(alg, 'loader_dict', None)
    if loader_dict is None:
        raise RuntimeError('alg.loader_dict 不存在：你当前 semilearn 版本的接口可能不同')

    train_lb_loader = loader_dict.get('train_lb', None)
    train_ulb_loader = loader_dict.get('train_ulb', None)
    eval_loader = loader_dict.get('eval', None)
    assert train_lb_loader is not None and train_ulb_loader is not None and eval_loader is not None
    print('✅ loaders ready')

    # 下面接你的 Trainer / logger / fit 逻辑

In [4]:
from collections import Counter

def get_targets(ds):
    for name in ['targets', 'labels', 'y']:
        if hasattr(ds, name):
            return list(getattr(ds, name))
    # 常见 torchvision ImageFolder 风格
    if hasattr(ds, 'samples'):
        return [y for _, y in ds.samples]
    if hasattr(ds, 'data') and hasattr(ds, 'targets'):
        return list(ds.targets)
    return None

ulb_ds = train_ulb_loader.dataset
lb_ds  = train_lb_loader.dataset
ev_ds  = eval_loader.dataset

for tag, ds in [('lb', lb_ds), ('ulb', ulb_ds), ('eval', ev_ds)]:
    t = get_targets(ds)
    if t is None:
        print(f'[{tag}] cannot find targets field, skip')
        continue
    c = Counter(t)
    print(f'[{tag}] size={len(t)}, dist={dict(sorted(c.items()))}')

# 强制：ulb 每类至少 1 张
t = get_targets(ulb_ds)
if t is not None:
    for k in range(cfg.num_classes):
        assert Counter(t).get(k, 0) > 0, f"❌ ULB class {k} is 0. Fix split first."
print("✅ split looks OK")

NameError: name 'train_ulb_loader' is not defined

## 3) 训练与评估（兼容不同 Trainer.fit 签名）

不同 semilearn 版本里 `Trainer.fit()` 有两种常见签名：
- `trainer.fit()` （Trainer 自己去拿 alg 内部 loader）
- `trainer.fit(train_lb_loader, train_ulb_loader, eval_loader)`

下面写法会自动兼容。

In [ ]:
from semilearn import Trainer, get_algorithm, get_net_builder
import os, logging, gc, torch

for i, cfg_dict in enumerate(run_configs, 1):
    cfg = get_config(cfg_dict)
    print(f"\n===== Run {i}/{len(run_configs)} =====")
    print(cfg.save_name)

    alg = get_algorithm(cfg, get_net_builder(cfg.net, from_name=False), tb_log=None, logger=None)

    # logger: 每个实验各自一个 log 文件
    log_dir = os.path.join(cfg.save_dir, cfg.save_name)
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(f"train_{cfg.save_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(logging.FileHandler(os.path.join(log_dir, "train.log")))
    logger.addHandler(logging.StreamHandler())

    if hasattr(alg, "logger"):
        alg.logger = logger

    trainer = Trainer(cfg, alg)
    if hasattr(trainer, "logger"):
        trainer.logger = logger

    # 禁止保存 pth（你已加 save_model=False，这里再保险）
    def _noop(*args, **kwargs):
        return None
    if hasattr(trainer, "save_model"):
        trainer.save_model = _noop
    if hasattr(trainer, "save_checkpoint"):
        trainer.save_checkpoint = _noop

    # 训练
    try:
        trainer.fit()
    except TypeError:
        loader_dict = getattr(alg, 'loader_dict', None)
        trainer.fit(loader_dict['train_lb'], loader_dict['train_ulb'], loader_dict['eval'])

    # 评估
    try:
        out = trainer.evaluate(alg.loader_dict['eval'])
        print('evaluate output:', out)
    except Exception as e:
        print('trainer.evaluate(eval_loader) failed:', repr(e))

    # 释放显存，避免长循环 OOM
    del trainer, alg
    gc.collect()
    torch.cuda.empty_cache()


## 4) 正式训练（把 iter/epoch 拉大）

当上面 2 epoch / 50 iter 能跑通后，再把这里的参数加大，例如：
- `epoch=20`
- `num_train_iter=2000`
- `num_eval_iter=200`

同时小数据集建议：
- `p_cutoff` 可以试试 0.7~0.95
- `uratio` 先用 1~2
- 如果过拟合很快，减小 lr 或增加 weight decay/数据增强